# 01 — Schools acquisition (Stage 1)

Per the project plan, this notebook builds the schools layer for the analysis. It:

1. Downloads the GIAS `edubasealldata` snapshot, registers its SHA256 in `data/manifest.csv`, and chmods it read-only.
2. Filters to the 12 NE England LAs, state-funded primary / secondary / special, status open.
3. Audits URN uniqueness, missing coordinates, and a coarse BNG bounding-box sanity check.
4. Saves the primary cohort as a GeoPackage with a provenance sidecar.
5. Saves the FE / sixth-form / independent rows in the NE as a separate sensitivity layer.

**Deferred to Stage 2:** strict point-in-polygon validation against LA boundary polygons (boundaries land in `02_geography_deprivation.ipynb`).

## Decisions referenced

- DEC-005 (EPSG:27700)
- DEC-009 (state-funded primary/secondary/special as primary cohort)
- The phase / type / status filter constants live in `schools_sunbeds.schools` so they are reviewable in one place.

## 0. Pre-flight: import package and verify manifest

Every analytic notebook re-verifies the manifest at the top so pipeline drift fails loudly.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import pandas as pd

from schools_sunbeds import audit, config, schools

config.ensure_dirs()
audit.verify_manifest()  # strict; raises on drift

TODAY = datetime.now(timezone.utc).strftime("%Y%m%d")
print("DATA_ROOT :", config.DATA_ROOT)
print("TODAY     :", TODAY)

## 1. Download the GIAS snapshot and register it

If auto-download fails (e.g. today's snapshot is not yet published or the URL pattern has changed), the function raises with an explicit instruction to download manually from the GIAS Downloads page and place the file at the path it expects. The hash is the same either way — only the network step differs.

In [ ]:
GIAS_DIR = config.DATA_RAW / "gias" / TODAY
csv_path = schools.fetch_gias_snapshot(GIAS_DIR)

audit.register_raw_file(
    csv_path,
    source_url="https://get-information-schools.service.gov.uk/Downloads",
    licence="Open Government Licence v3.0",
    notes=f"GIAS edubasealldata snapshot for {TODAY}",
)
print("Registered:", csv_path)

## 2. Load and filter to the in-scope cohort

`filter_in_scope` returns the in-scope rows plus an audit dictionary recording the row count after each filter step. We surface it in two views:

- A waterfall (initial → after LA → after phase-or-type-exempt → after type → after status) so attrition is visible. The "phase or type-exempt" step keeps Special schools whose GIAS `phase` is "Not applicable".
- A retained counts table by ONS LA code × phase, which we will later compare to public DfE statistics as a credibility check.

In [ ]:
raw = schools.load_gias_csv(csv_path)
print(f"Raw rows in GIAS: {len(raw):,}")

in_scope_df, audit_filter = schools.filter_in_scope(raw)

waterfall = pd.Series(
    {
        k: audit_filter[k]
        for k in (
            "initial_n",
            "after_la_n",
            "after_phase_or_exempt_n",
            "after_type_n",
            "after_status_n",
        )
    },
    name="n",
)
print("\nFilter waterfall:")
print(waterfall.to_string())

In [ ]:
per_la_phase = audit_filter["per_la_phase"]
per_la_phase.index = per_la_phase.index.map(lambda c: f"{c} {config.LA_NAMES_NE[c]}")
per_la_phase.loc["TOTAL"] = per_la_phase.sum()
per_la_phase["TOTAL"] = per_la_phase.sum(axis=1)
per_la_phase

## 3. Spatialise and run basic audits

We project to EPSG:27700 (BNG) directly — GIAS already publishes Easting / Northing — and then check:

- URN uniqueness (the URN is GIAS's primary key; duplicates would indicate a join error).
- That every point falls inside the loose NE bounding box (catches data corruption, not LA mis-assignment).
- Missing-coord rows are dropped at the GeoDataFrame conversion step and the count is reported.

In [ ]:
schools_gdf = schools.to_geodataframe(in_scope_df)
audit_gdf = schools.audit_basic(schools_gdf)

for k, v in audit_gdf.items():
    if k == "per_la_phase":
        continue
    print(f"{k:>22s}: {v}")

if audit_gdf["duplicate_urns"]:
    raise RuntimeError(f"Duplicate URNs detected: {audit_gdf['duplicate_urns']}")
if audit_gdf["n_outside_ne_bbox"]:
    print("WARN: schools outside NE bbox — investigate before continuing.")

## 4. Sensitivity layer (FE / sixth-form / independents in the NE)

Per spec §4 these are excluded from the primary cohort but kept for sensitivity analysis. We persist them separately so they can be loaded from a single layer in Stage 8.

In [ ]:
sens_df = schools.filter_sensitivity_layer(raw)
print(f"Sensitivity-layer rows: {len(sens_df):,}")
if len(sens_df):
    sens_gdf = schools.to_geodataframe(sens_df)
else:
    sens_gdf = gpd.GeoDataFrame(sens_df, geometry=[], crs=config.CRS_BNG)
sens_gdf["phase"].value_counts(dropna=False)

## 5. Write outputs and provenance sidecars

In [ ]:
out_primary = config.DATA_PROCESSED / f"schools_ne_{TODAY}.gpkg"
out_sens = config.DATA_PROCESSED / f"schools_ne_sensitivity_{TODAY}.gpkg"

schools_gdf.to_file(out_primary, driver="GPKG", layer="schools")
if len(sens_gdf):
    sens_gdf.to_file(out_sens, driver="GPKG", layer="schools_sensitivity")

audit.write_provenance_sidecar(
    audit.Provenance(
        output_path=out_primary,
        inputs=(csv_path,),
        notes="Stage 1 primary cohort — state-funded primary/secondary/special, status open, NE 12 LAs.",
        random_seed=config.RANDOM_SEED,
    )
)
if len(sens_gdf):
    audit.write_provenance_sidecar(
        audit.Provenance(
            output_path=out_sens,
            inputs=(csv_path,),
            notes="Stage 1 sensitivity layer — FE / sixth-form / independents in NE.",
            random_seed=config.RANDOM_SEED,
        )
    )

print("Wrote:", out_primary)
print("Wrote:", out_sens if len(sens_gdf) else "(no sensitivity rows)")

## 6. Quick-look map

Sanity check: do the points cluster where the conurbations are? Tyneside, Wearside, Teesside, and County Durham should be obvious.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 9))
schools_gdf.plot(ax=ax, column="phase", markersize=4, legend=True, alpha=0.7)
ax.set_title(f"NE schools (n={len(schools_gdf):,}) — GIAS snapshot {TODAY}")
ax.set_xlabel("Easting (m, BNG)")
ax.set_ylabel("Northing (m, BNG)")
plt.tight_layout()
plt.show()

## Done

Outputs registered in `data/processed/`:

- `schools_ne_<date>.gpkg` — primary cohort (Stages 4–7 read this).
- `schools_ne_sensitivity_<date>.gpkg` — sensitivity layer (Stage 8).

Strict point-in-polygon validation lands in Stage 2 once LA boundary polygons are loaded.